In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - requests: fetches raw HTML content from the target URL
#   - BeautifulSoup: parses HTML so we can search for specific elements
#   - csv: writes the extracted multi-column records to a CSV file

import csv

import requests
from bs4 import BeautifulSoup

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your scrape target and field definitions here before running any section below.
#
# url          - The page you want to scrape
# parent_tag   - The HTML tag that wraps each individual record on the page
# parent_class - CSS class to filter parent elements (set to None to match all)
# child_tag    - The tag to search for inside each parent element
# fields       - List of (column_name, attr_name, attr_value) tuples defining
#                what to extract from each record. Each tuple maps to one CSV column.
#                Example: ('brand_name', 'databind', 'text: name') will find
#                a child element where databind="text: name" and extract its text.
# output_path  - Where to save the resulting CSV

url          = 'https://www.cdms.net/Label-Database/Advanced-Search#Result-products'
parent_tag   = 'span'
parent_class = None   # e.g. 'product-tile', or None to match all parent_tag elements
child_tag    = 'span'
fields       = [
    ('brand_name',        'databind', 'text: name'),
    ('registration_number', 'databind', 'text: regNumber'),
    # Add more fields here as needed:
    # ('column_name', 'attr_name', 'attr_value'),
]
output_path  = 'active_ingredients.csv'

print(f"Config ready. Target URL: {url}")
print(f"Fields to extract: {[f[0] for f in fields]}")

In [ ]:
# %%
# SECTION 3: fetch_page()
# ------------------------
# Sends an HTTP GET request to the configured URL and returns raw page content.
# Sets a User-Agent header so the request looks like a normal browser visit.
# Raises an error and stops cleanly if the request fails (e.g. 404, timeout).
# Run this section first — all other sections depend on the 'page_content' variable.

def fetch_page(url: str) -> bytes:
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.content

try:
    page_content = fetch_page(url)
    print(f"Page fetched successfully. ({len(page_content)} bytes)")
except requests.RequestException as e:
    print(f"Error fetching page: {e}")
    page_content = None

In [ ]:
# %%
# SECTION 4: Parse HTML
# ----------------------
# Takes the raw HTML bytes from fetch_page() and parses them into a BeautifulSoup
# object ('soup'). Required before any extraction sections can run.
# Prints a short preview so you can confirm the page loaded the content you expect
# (some pages require JavaScript and won't show data here — see the note below).
#
# NOTE: If the preview shows little or no product data, the page may be
# JavaScript-rendered. In that case, try using Selenium or Playwright instead
# of requests to fetch the page after JS has executed.

if page_content:
    soup = BeautifulSoup(page_content, 'html.parser')
    print("HTML parsed successfully.")
    print("Preview (first 500 chars of text):")
    print(soup.get_text()[:500])
else:
    print("No page content to parse. Run Section 3 first.")

In [ ]:
# %%
# SECTION 5: Inspect parent elements
# ------------------------------------
# Finds all parent elements matching your configured parent_tag and parent_class
# and prints the first few so you can inspect their structure. Use this to verify
# you're targeting the right container before running the full extraction.
# Adjust parent_tag and parent_class in Section 2 if the results look wrong.

kwargs = {'class_': parent_class} if parent_class else {}
parents = soup.find_all(parent_tag, **kwargs)

print(f"Found {len(parents)} '{parent_tag}' elements.")
print("\nFirst 3 elements (raw HTML):")
for p in parents[:3]:
    print(p)
    print()

In [ ]:
# %%
# SECTION 6: scrape_records()
# ----------------------------
# Iterates over every parent element found in Section 5. For each one, searches
# for child elements matching each field definition in your 'fields' config.
# A field matches when a child element has the specified attribute set to the
# specified value (e.g. databind="text: name"). Extracts the text of each matched
# child and builds a dict per record. Skips records where all fields are empty.
# Results are stored in 'records' — a list of dicts, one per parent element.

def scrape_records(soup, parent_tag, parent_class, child_tag, fields):
    kwargs = {'class_': parent_class} if parent_class else {}
    parents = soup.find_all(parent_tag, **kwargs)

    records = []
    for parent in parents:
        record = {}
        for col_name, attr_name, attr_value in fields:
            child = parent.find(child_tag, attrs={attr_name: attr_value})
            record[col_name] = child.get_text(strip=True) if child else ''
        if any(record.values()):
            records.append(record)
    return records

records = scrape_records(soup, parent_tag, parent_class, child_tag, fields)

print(f"Extracted {len(records)} records.")
print("\nFirst 5 records:")
for r in records[:5]:
    print(r)

In [ ]:
# %%
# SECTION 7: write_csv()
# -----------------------
# Takes the 'records' list produced by Section 6 and writes it to a multi-column
# CSV at output_path. Column headers are taken from the field names defined in
# Section 2. Each record becomes one row. Prints a confirmation with the row count.
# Run this last, after confirming the Section 6 preview looks correct.

def write_csv(records, fieldnames, output_path):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"Wrote {len(records)} records to '{output_path}'")

if records:
    fieldnames = [f[0] for f in fields]
    write_csv(records, fieldnames, output_path)
else:
    print("No records to write. Run Section 6 first or check your field definitions.")